# Biopsy Location Breakdown (Supplementary Table S1)

Produces **Table S1**: for the same ICI-treated cohort as Table 1
(`11_demographic_table.ipynb`) and Tables S2, S4-S5, breaks down where the WGS
biopsy was taken from -- a full disclosure of the cohort's biopsy-site
composition, since the regression covariate derived from this field
(`biopsyStructure`, `BIOPSY_STRUCTURE_LEVELS` in `helpers/config.r`, used
in `5_models.ipynb`) collapses everything outside Lymph node/Liver/Lung/
Skin (subcutaneous/cutaneous) into a single "Other" bucket and has no
category for biopsies taken from the primary tumor itself.

Biopsy Location rows: **Primary tumor location**, **Liver**, **Lymph
node**, **Lung**, and **Skin (subcutaneous/cutaneous)** are shown
individually -- the same named set `BIOPSY_STRUCTURE_LEVELS`
(`helpers/config.r`) uses for the regression covariate (both pool the
"Skin (subcutaneous)" and "Skin (cutaneous)" raw values into one row),
plus "Primary tumor location", which that scheme folds into "Other". An
**Other** subtotal covers every remaining known biopsy site, and an
explicit **Unknown** row covers patients with no recorded biopsy site
(shown rather than silently excluded, for the same full-disclosure reason
as Tables S1, S2, S4, and S5). "Other" is immediately followed by a granular breakdown
of its own: a site gets its own row once it has at least
`SUBCOHORT_MIN_OTHER_TYPE_N` (`helpers/config.r`) ICI-treated patients;
rarer sites are pooled into a residual **"Other (Other)"** row. A final
"All biopsy locations" row gives the grand total.

This table works from `biopsyStructureRaw` -- the original
`clinical_go.txt` string, preserved unmodified by `3_meta.ipynb` -- rather
than the collapsed `biopsyStructure`/`BIOPSY_STRUCTURE_LEVELS` scheme, so
it can show the "Primary tumor location" category that scheme folds into
"Other". "Primary tumor location" is matched against
`BIOPSY_PRIMARY_TUMOR_RAW_VALUES` (`helpers/config.r`); extend that list
if `clinical_go.txt` records this under a different spelling.

Each row reports `n (%)` of the full ICI cohort -- there is no second
stratifying variable behind that column (unlike Tables S2, S4-S5), so "%" is
simply each Biopsy Location's share of the whole cohort, the same
convention as any single-variable row in Table 1/S1. "Other" and its
granular children (e.g. "Other (Other)") are a subtotal and its children,
not additional independent categories -- they should not be summed
together.

Two more columns, **Responders** and **Non-Responders**, report `n (row %)`
within each Biopsy Location row -- that site's own observed response rate
-- the same row-percentage convention Table S5 uses for its Responder/
Non-Responder columns. This is a different "%" base than the N (%) column
above (always a share of the whole cohort, not of the row).

Three more columns report the **CIDER T-cell Score** (`cider_score_depth_adj`,
Table 1's primary metric) split by response, within each Biopsy Location
row: median (IQR) for Responders, median (IQR) for Non-Responders, and a
two-sided Wilcoxon rank-sum p-value comparing the two -- the same
continuous-variable convention Table 1 uses (median (IQR); Wilcoxon
rank-sum p-value), applied here per biopsy site rather than across the
whole cohort, to see whether the Score's association with response holds
at each site. Patients with a missing CIDER T-cell Score are excluded from
these three columns only (the N (%) column above is unaffected); a row
with too few Responders or Non-Responders for a two-sample comparison
shows "—" instead of a p-value.

**Input:** `cpi_ready.rds`
**Output:**
`figs/table_s1_biopsy_location_breakdown.csv` -- kept purely as the intermediate
`assemble_supplementary_tables_excel.R` reads to build the combined
`supplementary_tables.xlsx`; no standalone `.html`/`.docx` are generated for
this table (see the combined workbook instead).


In [ ]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))  # tidyverse + fct_pin_last()
source(paste0(dirname(getwd()), "/helpers/config.r"))

dir.create(paste0(CIDER_PROJ_DIR, "figs/"), showWarnings = FALSE, recursive = TRUE)


In [ ]:
# ── Load ICI-ready data ───────────────────────────────────────────────────────
# Same population as Table 1 and Tables S2, S4-S5: ICI-treated patients with
# known RECIST response, one row per patient.
#
# IMPORTANT: cpi_ready.rds stores one row per (sampleId, gp) *and* a
# duplicated copy of every row with cohort_broad overwritten to
# "ICI-Treated" (added in 4_base.ipynb so Figure 3 can facet pooled-vs-per-
# cancer-type together). The gp filter below removes the first
# duplication; distinct() removes the second.
df_raw <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  filter(gp == "T-cell", !is.na(response)) |>
  distinct(sampleId, .keep_all = TRUE)


In [ ]:
# ── Variable preparation ──────────────────────────────────────────────────────
# Biopsy Location rows: five named categories (Primary tumor location,
# Liver, Lymph node, Lung, Skin (subcutaneous/cutaneous) -- the latter
# pools both the "Skin (subcutaneous)" and "Skin (cutaneous)" raw values
# into one row), an "Other" subtotal for every remaining known biopsy
# site (nested breakdown below), and an explicit "Unknown" row for
# patients with no recorded biopsy site -- shown rather than silently
# excluded, for the same full-disclosure reason as Tables S1, S2, S4, and S5.
#
# This works from biopsyStructureRaw (3_meta.ipynb), the original
# clinical_go.txt string, rather than the collapsed BIOPSY_STRUCTURE_LEVELS
# scheme (helpers/config.r) used as a regression covariate elsewhere --
# that scheme has no "Primary tumor location" category of its own (it
# folds these into "Other"), which is exactly the granularity this table
# exists to restore.
demo <- df_raw |>
  mutate(
    biopsy_broad = case_when(
      is.na(biopsyStructureRaw) ~ "Unknown",
      biopsyStructureRaw %in% BIOPSY_PRIMARY_TUMOR_RAW_VALUES ~ "Primary tumor location",
      biopsyStructureRaw == "Liver"                 ~ "Liver",
      biopsyStructureRaw == "Lymph node"             ~ "Lymph node",
      biopsyStructureRaw == "Lung"                   ~ "Lung",
      biopsyStructureRaw %in% c("Skin (subcutaneous)", "Skin (cutaneous)") ~ "Skin (subcutaneous/cutaneous)",
      TRUE ~ "Other"
    ),
    biopsy_broad = fct_pin_last(factor(biopsy_broad), c("Other", "Unknown")),
    # Same fields/derivation as Table 1 (11_demographic_table.ipynb), so the
    # CIDER T-cell Score columns below are directly comparable to Table 1's
    # "CIDER T-cell score" row.
    Responder             = factor(responder, levels = c("Responder", "Non-Responder")),
    `CIDER T-cell score`  = as.numeric(cider_score_depth_adj)
  )

# ── "Other" bucket breakdown ───────────────────────────────────────────────────
# Every raw biopsy site that isn't Primary tumor location/Liver/Lymph
# node/Lung/Skin (subcutaneous or cutaneous -- pooled into one
# "Skin (subcutaneous/cutaneous)" row) gets its own row once it has at
# least SUBCOHORT_MIN_OTHER_TYPE_N (helpers/config.r) ICI-treated
# patients; rarer sites are pooled into a residual "Other (Other)" row --
# the same nested-subtotal design used for the Tumor Type axis in
# Tables S2 and S4.
other_site_counts <- demo |>
  filter(biopsy_broad == "Other") |>
  count(biopsyStructureRaw, name = "n")

well_represented_sites <- other_site_counts |>
  filter(n >= SUBCOHORT_MIN_OTHER_TYPE_N) |>
  pull(biopsyStructureRaw)

demo <- demo |>
  mutate(
    biopsy_other_detail = case_when(
      biopsy_broad != "Other" ~ NA_character_,
      biopsyStructureRaw %in% well_represented_sites ~ paste0(biopsyStructureRaw, " (Other)"),
      TRUE ~ "Other (Other)"
    ),
    biopsy_other_detail = fct_pin_last(factor(biopsy_other_detail), "Other (Other)")
  )

message("\"Other\" bucket breakdown (sites with N >= ", SUBCOHORT_MIN_OTHER_TYPE_N, " shown individually):")
print(levels(demo$biopsy_other_detail))


In [ ]:
# ── Build one-way summary: N (%) of the ICI cohort per Biopsy Location ────────
# Unlike Tables S2, S4-S5, this table has a single axis -- no second
# stratifying variable was requested -- so "%" is simply each row's share
# of the whole ICI cohort (n_total below), the same convention as any
# single-variable row in Table 1/S3 (e.g. Sex).
#
# "Other" is a subtotal spanning the granular rows nested beneath it (not
# meant to be summed with them); "Unknown" is shown as its own row rather
# than excluded, so every patient in the cohort is accounted for
# somewhere and "All biopsy locations" sums to the full cohort.
n_total <- nrow(demo)

summarise_locations <- function(df, location_col) {
  df |>
    filter(!is.na({{ location_col }})) |>
    count(`Biopsy Location` = {{ location_col }}, name = "n") |>
    mutate(`N (%)` = sprintf("%d (%d%%)", n, round(100 * n / n_total))) |>
    select(`Biopsy Location`, `N (%)`)
}

# ── Responder / Non-Responder counts, per Biopsy Location row ─────────────────
# n (row %): within each Biopsy Location row, how many of that row's own
# patients were Responders vs Non-Responders -- i.e. that site's observed
# response rate, the same row-percentage convention Table S5 uses for its
# Responder/Non-Responder columns. This is a different "%" base than the
# N (%) column above (which is always a share of the whole cohort).
summarise_responder_counts <- function(df, location_col) {
  df |>
    filter(!is.na({{ location_col }})) |>
    group_by(`Biopsy Location` = {{ location_col }}) |>
    summarise(
      n_row          = dplyr::n(),
      n_responder    = sum(Responder == "Responder"),
      n_nonresponder = sum(Responder == "Non-Responder"),
      .groups        = "drop"
    ) |>
    mutate(
      Responders        = sprintf("%d (%d%%)", n_responder, round(100 * n_responder / n_row)),
      `Non-Responders`  = sprintf("%d (%d%%)", n_nonresponder, round(100 * n_nonresponder / n_row))
    ) |>
    select(`Biopsy Location`, Responders, `Non-Responders`)
}

# ── CIDER T-cell Score by response, per Biopsy Location row ───────────────────
# Same continuous-variable convention as Table 1: median (IQR), 0 decimals
# (matching Table 1's `all_continuous() ~ 0` digit rule for this metric),
# and a two-sided Wilcoxon rank-sum p-value. Computed within each Biopsy
# Location row rather than once for the whole cohort, to see whether the
# Score's association with response holds at each site. A row with zero
# Responders or zero Non-Responders can't support a two-sample comparison
# -- fmt_median_iqr()/safe_wilcox_p() return "—" rather than erroring.
fmt_median_iqr <- function(x) {
  x <- x[!is.na(x)]
  if (length(x) == 0) return("—")
  qs <- stats::quantile(x, c(0.5, 0.25, 0.75), type = 7)
  sprintf("%.0f (%.0f–%.0f)", qs[1], qs[2], qs[3])
}

safe_wilcox_p <- function(x, y) {
  x <- x[!is.na(x)]
  y <- y[!is.na(y)]
  tryCatch(stats::wilcox.test(x, y)$p.value, error = function(e) NA_real_)
}

fmt_pvalue <- function(p) {
  if (is.na(p)) "—" else if (p < 0.001) "<0.001" else sprintf("%.3f", p)
}

summarise_cider_by_response <- function(df, location_col) {
  df |>
    filter(!is.na({{ location_col }})) |>
    group_by(`Biopsy Location` = {{ location_col }}) |>
    summarise(
      `CIDER T-cell Score (Responders)` =
        fmt_median_iqr(`CIDER T-cell score`[Responder == "Responder"]),
      `CIDER T-cell Score (Non-Responders)` =
        fmt_median_iqr(`CIDER T-cell score`[Responder == "Non-Responder"]),
      `p-value` = fmt_pvalue(safe_wilcox_p(
        `CIDER T-cell score`[Responder == "Responder"],
        `CIDER T-cell score`[Responder == "Non-Responder"]
      )),
      .groups = "drop"
    )
}

location_order <- c(
  setdiff(levels(demo$biopsy_broad), c("Other", "Unknown")),
  "Other",
  levels(demo$biopsy_other_detail),
  "Unknown",
  "All biopsy locations"
)

n_grand_responder    <- sum(demo$Responder == "Responder")
n_grand_nonresponder <- sum(demo$Responder == "Non-Responder")

grand_total <- tibble(
  `Biopsy Location` = "All biopsy locations",
  `N (%)` = sprintf("%d (100%%)", n_total),
  Responders = sprintf("%d (%d%%)", n_grand_responder, round(100 * n_grand_responder / n_total)),
  `Non-Responders` = sprintf("%d (%d%%)", n_grand_nonresponder, round(100 * n_grand_nonresponder / n_total)),
  `CIDER T-cell Score (Responders)` =
    fmt_median_iqr(demo$`CIDER T-cell score`[demo$Responder == "Responder"]),
  `CIDER T-cell Score (Non-Responders)` =
    fmt_median_iqr(demo$`CIDER T-cell score`[demo$Responder == "Non-Responder"]),
  `p-value` = fmt_pvalue(safe_wilcox_p(
    demo$`CIDER T-cell score`[demo$Responder == "Responder"],
    demo$`CIDER T-cell score`[demo$Responder == "Non-Responder"]
  ))
)

wide <- bind_rows(
  summarise_locations(demo, biopsy_broad) |>
    left_join(summarise_responder_counts(demo, biopsy_broad), by = "Biopsy Location") |>
    left_join(summarise_cider_by_response(demo, biopsy_broad), by = "Biopsy Location"),
  summarise_locations(demo, biopsy_other_detail) |>
    left_join(summarise_responder_counts(demo, biopsy_other_detail), by = "Biopsy Location") |>
    left_join(summarise_cider_by_response(demo, biopsy_other_detail), by = "Biopsy Location"),
  grand_total
) |>
  mutate(`Biopsy Location` = factor(`Biopsy Location`, levels = location_order)) |>
  arrange(`Biopsy Location`)

wide


In [ ]:
# ── Save outputs ──────────────────────────────────────────────────────────────
out_dir <- paste0(CIDER_PROJ_DIR, "figs/")

n_unknown      <- sum(demo$biopsy_broad == "Unknown")
n_cider_missing <- sum(is.na(demo$`CIDER T-cell score`))

footnote_text <- sprintf(
  paste0(
    "All %d ICI-treated patients with known RECIST response (Table 1's cohort) are ",
    "included; %d (%d%%) have no recorded biopsy site and appear as \"Unknown\" rather ",
    "than being excluded. Percentages in the N (%%) column are of the full cohort ",
    "(N = %d); percentages in the Responders/Non-Responders columns are of that row's ",
    "own N instead -- each site's own observed response rate. \"Other\" is a subtotal ",
    "spanning the rows below it; it should not be summed with its children or with ",
    "\"All biopsy locations\". CIDER T-cell Score columns report median (IQR) within ",
    "Responders/Non-Responders and a two-sided Wilcoxon rank-sum p-value comparing ",
    "them, per row; %d patient(s) with a missing CIDER T-cell Score are excluded from ",
    "these three columns only. A row with too few Responders or Non-Responders for a ",
    "two-sample comparison shows \"—\" instead of a p-value."
  ),
  n_total, n_unknown, round(100 * n_unknown / n_total), n_total, n_cider_missing
)

# CSV — build intermediate only. Not a standalone deliverable: this
# table's actual output is its sheet in the combined
# supplementary_tables.xlsx (assemble_supplementary_tables_excel.R reads
# this .csv + the metadata sidecar below to build it).
wide |>
  readr::write_csv(paste0(out_dir, "table_s1_biopsy_location_breakdown.csv"))

message("Saved: table_s1_biopsy_location_breakdown.csv")

# Metadata sidecar (title + footnote) -- consumed by
# assemble_supplementary_tables_excel.R to build the combined supplemental
# tables Excel workbook without re-deriving this table's title/footnote
# text (footnote_text is computed above).
jsonlite::write_json(
  list(
    title    = "Supplementary Table S1. Biopsy location breakdown (ICI cohort)",
    footnote = footnote_text
  ),
  paste0(out_dir, "table_s1_biopsy_location_breakdown_meta.json"),
  auto_unbox = TRUE
)
